# Lesson 6 | What is RTL?

Boolean relationships are now familiar. We need to describe inputs, outputs, stored state, and update timing. Today asks:
> **How can code-like text describe digital hardware rather than a program for a CPU to execute line by line?**

Primary concept: **Register-Transfer Level (RTL)**.


## 1. Concept ledger

Lesson 5 explained clock event control, synchronous reset, and nonblocking assignment. Here we study RTL as a design level, module boundaries and ports, then compile the accumulator with Icarus Verilog.


## 2. Four terms

**Hardware Description Language (HDL):** a language class for describing digital hardware structure and behavior.

**SystemVerilog:** the HDL and verification language used in this project.

**RTL:** a level focused on what registers hold and how data is computed/transferred across clock cycles.

**module / port:** a module is a bounded hardware unit; ports are signals crossing that boundary.


## 3. HDL may look like software while meaning something different

Adjacent Python statements usually imply execution order. RTL often describes hardware relationships existing at the same time. First ask: what are the inputs, outputs, stored state, and update edge?


## 4. Read one already-known behavior: a clocked accumulator

It implements the same rule as the Lesson 4 accumulator, now in SystemVerilog:

```systemverilog
module clocked_accumulator (
    input  logic              clk,
    input  logic              rst_n,
    input  logic signed [7:0] input_value,
    output logic signed [7:0] state
);
    always_ff @(posedge clk) begin
        if (!rst_n)
            state <= '0;
        else
            state <= state + input_value;
    end
endmodule
```


## 5. Read signal declarations

`module` and `endmodule` delimit the module. The author chooses `clocked_accumulator`. Inputs are `clk`, `rst_n`, and `input_value`; the output is `state`. Plain `logic` is one bit, while `logic signed [7:0]` declares eight signed bits.

`always_ff`, `posedge`, `input`, and `output` are standard keywords. `clk` and `state` are user names. The event control triggers at a positive clock edge; the nonblocking assignment evaluates the old state and updates in the current time step. Lesson 5 provides a token-by-token explanation and timing table.

An eight-bit signed value ranges from −128 to 127. In this example, `127 + 1` retains `10000000`, interpreted as −128. Whether to use wraparound or saturation is a numerical specification decision.


## 6. Reset is part of the contract

The `_n` suffix in `rst_n` indicates active-low reset. This example uses synchronous reset, so reset changes state only at a clock edge. Polarity and timing must be explicit.


## 7. Run: compile the module

**Icarus Verilog** compiles and simulates HDL. `-g2012` selects SystemVerilog 2012 and `-tnull` checks syntax and elaborates without generating a simulation executable. Python `subprocess.run(..., check=True)` runs the external tool and reports a failing exit status.

Predict whether removing a port separator is a syntax error or an accumulation behavior error. Compile the unchanged file here.


In [ ]:
from pathlib import Path
import shutil, subprocess

def repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p/'pyproject.toml').exists() and (p/'lessons').exists():
            return p
    raise FileNotFoundError('Run inside FPGA-FlyBrain')

root = repo_root()
iverilog = shutil.which('iverilog')
if not iverilog:
    print('Icarus Verilog not found; RTL compile check did not run.')
else:
    subprocess.run([iverilog, '-g2012', '-tnull', str(root/'rtl/learning/clocked_accumulator.sv')], check=True)
    print('COMPILE PASS: clocked_accumulator.sv')


## 8. Observe

`COMPILE PASS` establishes syntax/basic elaboration only. It does not prove inputs `1,2,3` produce states `1,3,6`. That gap motivates the later testbench lesson.


## 9. Try It

Identify three input ports and one output, the stored signal, clock event, and reset branch. Calculate the next edge for `state=3, input_value=2, rst_n=1`; explain why compilation has not checked that result.


## Exercise

[Lesson 6 exercise: translate one RTL clock edge into Python semantics](../../exercises/en/06_what_is_rtl.ipynb)

The standalone workbook checks this lesson's semantic understanding; the real RTL/testbench experiment in the lesson remains part of the course.

## 10. AI Task

Ask AI only to annotate the existing module: boundary, ports, stored state, and clocked update. Do not let it redesign the code.


## 11. Human Check

Without AI, explain HDL vs ordinary software semantics, module vs port, what `always_ff @(posedge clk)` tells you, and why compile success is not behavior verification.


## 12. Engineering Handoff

`rtl/learning/clocked_accumulator.sv` is a teaching artifact. Next lesson decomposes a known neuron contract into a combinational path plus sequential register update.


## 13. Project Trace

- Lesson: `LSN-006`
- Prepares: `RMD-004`
- Teaching RTL: `rtl/learning/clocked_accumulator.sv`


## 14. Exit Ticket

You can explain RTL / HDL / SystemVerilog / module / port, read the state and update edge in a minimal clocked module, and avoid confusing “compiled” with “verified.”
